# 01b — Fallback data path: spin-spiral labels + per-material structures
**Project:** MAG2D-NC | **Phase:** F4 | **Protocol:** v1.0 (frozen) | Plan C

Used while the bulk C2DB file is pending. Two stages:
1. **Labels (D2):** parse the 164-material ground-state table from the open-access
   supplementary of Sødequist & Olsen, npj Comput. Mater. 10, 170 (2024).
2. **Structures (D1 subset):** fetch each material's structure file from the C2DB
   web interface — politely (rate-limited, cached, resumable).

Honesty rules preserved: nothing is fabricated; every stage ends in a
verification gate (class counts 58 / 21 / 85, DM_SS = 15) and hard-stops on
mismatch. When the bulk file arrives, notebook 01 supersedes this one; outputs
are format-compatible.

## CONFIG

In [ ]:
from pathlib import Path
from datetime import datetime

CONFIG = {
    "PROJECT_ROOT": Path.home() / "MAG2D-NC",
    # Download the supplementary PDF manually from the article page (open access):
    # https://doi.org/10.1038/s41524-024-01318-2  -> "Supplementary information"
    "SUPP_PDF": Path.home() / "MAG2D-NC" / "dataset" / "sodequist2024_supplementary.pdf",
    # Structure download URL template — CONFIRM IN BROWSER (see markdown below)
    # Example placeholder: "https://c2db.fysik.dtu.dk/material/{uid}/download/structure.json"
    "STRUCT_URL_TEMPLATE": None,
    "STRUCT_DIR": Path.home() / "MAG2D-NC" / "dataset" / "structures_164",
    "POLITE_DELAY_S": 2.0,          # seconds between requests — do not lower
    "EXPECTED_COUNTS": {"FM": 58, "AFM_collinear": 21, "NC_total": 85, "DM_SS": 15},
    "RUN_STAMP": datetime.now().strftime("%Y%m%d-%H%M%S"),
}
CONFIG["STRUCT_DIR"].mkdir(parents=True, exist_ok=True)
print({k: str(v) for k, v in CONFIG.items()})

## Dependencies
`pdfplumber` for the supplementary table, `requests` for downloads, `ase` to
validate structures.

In [ ]:
import importlib, subprocess, sys
for pkg in ["pdfplumber", "requests", "ase", "pandas"]:
    try:
        importlib.import_module(pkg); print(pkg, "OK")
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", pkg], check=True)
        print(pkg, "installed")

## Stage 1 — Parse the supplementary tables
The supplementary lists every material with its ground-state classification
(ordering vector / FM / collinear AFM / non-collinear; DM spirals in the main
paper's Table 1). PDF table layouts are fragile, so this cell extracts raw text
first and prints a sample — inspect it before trusting the parser.

In [ ]:
import pdfplumber

assert CONFIG["SUPP_PDF"].exists(), (
    f"Supplementary PDF not found at {CONFIG['SUPP_PDF']}.\n"
    "Download it from https://doi.org/10.1038/s41524-024-01318-2 (open access) "
    "and place it there."
)

pages_text = []
with pdfplumber.open(CONFIG["SUPP_PDF"]) as pdf:
    print(f"Supplementary has {len(pdf.pages)} pages")
    for p in pdf.pages:
        pages_text.append(p.extract_text() or "")

# verification output: locate candidate table pages
for i, t in enumerate(pages_text):
    head = t[:90].replace("\n", " | ")
    print(f"p{i+1:02d}: {head}")

## Stage 1b — Table extraction
After inspecting the page dump above, set `TABLE_PAGES` to the pages holding the
ground-state table, then run. The parser expects lines of the form
`<Formula> <Qx> <Qy> ...`; adjust the regex ONLY if the sample lines demand it,
and record any change in the lab log.

In [ ]:
import re
import pandas as pd

TABLE_PAGES = None   # e.g. [3, 4, 5] — 1-indexed; fill after inspection
assert TABLE_PAGES is not None, "Set TABLE_PAGES after inspecting the page dump."

line_re = re.compile(
    r"^(?P<formula>[A-Z][A-Za-z0-9]*)\s+"
    r"(?P<qx>-?\d*\.?\d+(?:/\d+)?)\s*[, ]\s*(?P<qy>-?\d*\.?\d+(?:/\d+)?)"
)

def parse_frac(s):
    return eval(s) if "/" in s else float(s)     # noqa: S307 — controlled input

rows = []
for pno in TABLE_PAGES:
    for line in pages_text[pno - 1].splitlines():
        m = line_re.match(line.strip())
        if m:
            rows.append({
                "formula": m["formula"],
                "qx": parse_frac(m["qx"]),
                "qy": parse_frac(m["qy"]),
                "raw": line.strip(),
            })

labels = pd.DataFrame(rows)
print(f"Parsed {len(labels)} candidate rows")
print(labels.head(10).to_string())
# Manual-review dump regardless of success:
review = CONFIG["PROJECT_ROOT"] / "output" / f"supp_parse_review_{CONFIG['RUN_STAMP']}.csv"
labels.to_csv(review, index=False)
print("Review file written:", review.name)

## Stage 1c — Label construction + verification gate
Same frozen mapping as notebook 01. DM spin spirals: the 15 materials in the
main paper's Table 1 (typed in verbatim below from the published table — this is
transcription of published data, not invention).

In [ ]:
import numpy as np

DM_SS_FORMULAS = ["CoBr2","NiBr2","TaO2","CrSeI","PdHfCl6","NiHfBr6","NiHfI6",
                  "NiZrBr6","NiZrI6","VF2O","VCuP2Se6","VAgP2Se6","VAuP2Se6",
                  "VClIO","ReAu2F6"]   # Table 1, Sodequist & Olsen (2024)

def classify(qx, qy, formula):
    q = np.mod(np.array([qx, qy]) + 0.5, 1.0) - 0.5
    if formula in DM_SS_FORMULAS:
        return "DM_SS"
    if np.allclose(q, 0.0, atol=1e-3):
        return "FM"
    if all(np.isclose(abs(c), 0.5, atol=1e-3) or np.isclose(c, 0.0, atol=1e-3) for c in q) \
       and not np.allclose(q, 0.0, atol=1e-3):
        return "AFM_collinear"
    return "NC"

labels["label4"] = [classify(r.qx, r.qy, r.formula) for r in labels.itertuples()]
labels["label2"] = labels["label4"].map({"FM":"collinear","AFM_collinear":"collinear",
                                         "NC":"non_collinear","DM_SS":"non_collinear"})
counts = labels["label4"].value_counts().to_dict()
print("Counts:", counts)

exp = CONFIG["EXPECTED_COUNTS"]
ok = (counts.get("FM",0)==exp["FM"] and counts.get("AFM_collinear",0)==exp["AFM_collinear"]
      and counts.get("NC",0)+counts.get("DM_SS",0)==exp["NC_total"]
      and counts.get("DM_SS",0)==exp["DM_SS"])
assert ok, ("Verification gate FAILED — expected 58/21/85(15). "
            "Inspect the review CSV; likely a parsing gap. Do not proceed.")
out = CONFIG["PROJECT_ROOT"] / "dataset" / f"spiral_labels_{CONFIG['RUN_STAMP']}.parquet"
labels.to_parquet(out, index=False)
print("GATE PASSED. Saved:", out.name)

## Stage 2 — Confirm the structure download URL (one manual step)
Open one material page in your browser, e.g. search `NiBr2` at
https://c2db.fysik.dtu.dk/ , right-click its structure **Download** button and
*Copy link address*. Paste it below with the material-specific part replaced by
`{uid}` (and note what the uid looks like — formula-based or numeric). This
single confirmed example anchors the whole batch download; we never guess URLs.

In [ ]:
# Paste ONE confirmed example first, e.g.:
# EXAMPLE_CONFIRMED = "https://c2db.fysik.dtu.dk/material/1NiBr2-1/download/structure.xyz"
EXAMPLE_CONFIRMED = None
assert EXAMPLE_CONFIRMED, "Paste a browser-confirmed download link first."
print("Anchor link:", EXAMPLE_CONFIRMED)
# Derive template by replacing the uid segment manually:
CONFIG["STRUCT_URL_TEMPLATE"] = None   # e.g. ".../material/{uid}/download/structure.xyz"
assert CONFIG["STRUCT_URL_TEMPLATE"], "Set STRUCT_URL_TEMPLATE from the anchor link."
print("Template:", CONFIG["STRUCT_URL_TEMPLATE"])

## Stage 2b — Polite, cached, resumable batch fetch
2 s between requests (~6 min for 164 files), skips files already on disk,
validates each with ASE, logs failures to a CSV instead of crashing.

In [ ]:
import time, requests
from ase.io import read as ase_read

failures, fetched, cached = [], 0, 0
uids = labels["uid"].tolist() if "uid" in labels.columns else labels["formula"].tolist()

for uid in uids:
    dest = CONFIG["STRUCT_DIR"] / f"{uid}{Path(CONFIG['STRUCT_URL_TEMPLATE']).suffix}"
    if dest.exists():
        cached += 1
        continue
    url = CONFIG["STRUCT_URL_TEMPLATE"].format(uid=uid)
    try:
        r = requests.get(url, timeout=30,
                         headers={"User-Agent": "MAG2D-NC academic research (contact: your.email)"})
        r.raise_for_status()
        dest.write_bytes(r.content)
        ase_read(dest)                      # validation: parse or raise
        fetched += 1
    except Exception as e:
        failures.append({"uid": uid, "url": url, "error": repr(e)})
        if dest.exists():
            dest.unlink()
    time.sleep(CONFIG["POLITE_DELAY_S"])

print(f"fetched={fetched} cached={cached} failed={len(failures)} / {len(uids)}")
if failures:
    import pandas as pd
    fpath = CONFIG["PROJECT_ROOT"] / "output" / f"fetch_failures_{CONFIG['RUN_STAMP']}.csv"
    pd.DataFrame(failures).to_csv(fpath, index=False)
    print("Failure log:", fpath.name, "- resolve uid mapping for these, then rerun (resumable).")
assert len(failures) == 0, "Some structures failed — inspect the failure log before proceeding."
print("All 164 structures on disk and ASE-valid.")

## Done
Outputs: `spiral_labels_*.parquet` + `structures_164/`. These feed the same
group-ID and data-card cells as notebook 01 (run those there, pointing at these
files). When the bulk c2db.db arrives, rerun notebook 01 end-to-end and diff the
label table against this one — any discrepancy is reported, not silently
resolved.